In [1]:
import json

primary_none = 0
secondary_none = 0
priority_none = 0
reason_none = 0

with open("./v5a/V5A.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        output = data.get("output", {})

        if str(output.get("primary_objective", "")).strip().lower() == "none":
            primary_none += 1

        if str(output.get("secondary_objective", "")).strip().lower() == "none":
            secondary_none += 1

        if str(output.get("priority", "")).strip().lower() == "none":
            priority_none += 1

        if str(output.get("reason", "")).strip().lower() == "none":
            reason_none += 1

print("Primary Objective None:", primary_none)
print("Secondary Objective None:", secondary_none)
print("Priority None:", priority_none)
print("Reason None:", reason_none)

Primary Objective None: 0
Secondary Objective None: 9442
Priority None: 0
Reason None: 0


V5B


In [2]:
import json

# ==========================================================
# V5B NULL / BLANK VALUE CHECK
# ==========================================================

FILE =  r"./V5B/V5B.jsonl"

FIELDS = [
    "tone",
    "communication_style",
    "detail_level",
    "approach"
]

# Counters
total_records = 0
invalid_json = 0

stats = {
    field: {
        "missing": 0,
        "null": 0,
        "blank": 0,
        "none_string": 0,
        "valid": 0
    }
    for field in FIELDS
}

# Store problematic records
problem_records = []


# ==========================================================
# READ DATASET
# ==========================================================

with open(FILE, "r", encoding="utf-8") as f:

    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        total_records += 1

        # --------------------------------------------------
        # Check JSON
        # --------------------------------------------------

        try:
            data = json.loads(line)

        except json.JSONDecodeError as e:

            invalid_json += 1

            problem_records.append({
                "line": line_number,
                "problem": "Invalid JSON"
            })

            continue

        output = data.get("output")

        # --------------------------------------------------
        # Check output object
        # --------------------------------------------------

        if not isinstance(output, dict):

            for field in FIELDS:
                stats[field]["missing"] += 1

            problem_records.append({
                "line": line_number,
                "problem": "output is missing or not an object"
            })

            continue

        # --------------------------------------------------
        # Check each V5B field
        # --------------------------------------------------

        for field in FIELDS:

            if field not in output:

                stats[field]["missing"] += 1

                problem_records.append({
                    "line": line_number,
                    "field": field,
                    "problem": "missing"
                })

                continue

            value = output[field]

            # NULL
            if value is None:

                stats[field]["null"] += 1

                problem_records.append({
                    "line": line_number,
                    "field": field,
                    "value": None,
                    "problem": "null"
                })

            # STRING
            elif isinstance(value, str):

                cleaned = value.strip()

                # BLANK
                if cleaned == "":

                    stats[field]["blank"] += 1

                    problem_records.append({
                        "line": line_number,
                        "field": field,
                        "value": value,
                        "problem": "blank"
                    })

                # "None"
                elif cleaned.lower() == "none":

                    stats[field]["none_string"] += 1

                    problem_records.append({
                        "line": line_number,
                        "field": field,
                        "value": value,
                        "problem": "None string"
                    })

                # VALID
                else:

                    stats[field]["valid"] += 1

            # WRONG TYPE
            else:

                stats[field]["blank"] += 1

                problem_records.append({
                    "line": line_number,
                    "field": field,
                    "value": value,
                    "problem": f"invalid type: {type(value).__name__}"
                })


# ==========================================================
# PRINT SUMMARY
# ==========================================================

print("\n" + "=" * 75)
print("V5B DATASET NULL / BLANK CHECK")
print("=" * 75)

print(f"\nTotal records     : {total_records:,}")
print(f"Invalid JSON      : {invalid_json:,}")

print("\n" + "-" * 75)

for field in FIELDS:

    s = stats[field]

    problematic = (
        s["missing"]
        + s["null"]
        + s["blank"]
        + s["none_string"]
    )

    print(f"\n{field}")
    print(f"  Valid        : {s['valid']:,}")
    print(f"  Missing      : {s['missing']:,}")
    print(f"  Null         : {s['null']:,}")
    print(f"  Blank        : {s['blank']:,}")
    print(f"  'None' text  : {s['none_string']:,}")
    print(f"  Problematic  : {problematic:,}")

    if total_records > 0:
        print(
            f"  Problem %    : "
            f"{problematic / total_records * 100:.2f}%"
        )


# ==========================================================
# TOTAL PROBLEMATIC RECORDS
# ==========================================================

problem_line_numbers = set()

for item in problem_records:

    if "line" in item:
        problem_line_numbers.add(item["line"])


print("\n" + "=" * 75)
print("OVERALL RESULT")
print("=" * 75)

print(
    f"Records with at least one problem : "
    f"{len(problem_line_numbers):,}"
)

if total_records > 0:

    print(
        f"Percentage affected              : "
        f"{len(problem_line_numbers) / total_records * 100:.2f}%"
    )


# ==========================================================
# SHOW FIRST 30 PROBLEMS
# ==========================================================

print("\n" + "=" * 75)
print("FIRST 30 PROBLEMATIC VALUES")
print("=" * 75)

for item in problem_records[:30]:

    print(item)


# ==========================================================
# FINAL STATUS
# ==========================================================

print("\n" + "=" * 75)
print("FINAL STATUS")
print("=" * 75)

if len(problem_line_numbers) == 0:

    print("✅ No null, blank, missing, or 'None' values found.")

else:

    print(
        f"⚠️ Found {len(problem_line_numbers):,} "
        "records containing problematic V5B output values."
    )


V5B DATASET NULL / BLANK CHECK

Total records     : 65,385
Invalid JSON      : 0

---------------------------------------------------------------------------

tone
  Valid        : 65,385
  Missing      : 0
  Null         : 0
  Blank        : 0
  'None' text  : 0
  Problematic  : 0
  Problem %    : 0.00%

communication_style
  Valid        : 65,385
  Missing      : 0
  Null         : 0
  Blank        : 0
  'None' text  : 0
  Problematic  : 0
  Problem %    : 0.00%

detail_level
  Valid        : 65,385
  Missing      : 0
  Null         : 0
  Blank        : 0
  'None' text  : 0
  Problematic  : 0
  Problem %    : 0.00%

approach
  Valid        : 65,385
  Missing      : 0
  Null         : 0
  Blank        : 0
  'None' text  : 0
  Problematic  : 0
  Problem %    : 0.00%

OVERALL RESULT
Records with at least one problem : 0
Percentage affected              : 0.00%

FIRST 30 PROBLEMATIC VALUES

FINAL STATUS
✅ No null, blank, missing, or 'None' values found.


In [1]:
import json

FILE = r"G:\V5-dataset\V5C\V5C.jsonl"

EXPECTED = {
    "complete": 0.80,
    "partial": 0.15,
    "little_none": 0.05
}

counts = {
    "complete": 0,
    "partial": 0,
    "little_none": 0,
    "invalid": 0
}

total = 0


def count_turns(conversation):
    if not isinstance(conversation, str):
        return 0

    lines = [
        x for x in conversation.splitlines()
        if ":" in x and x.split(":", 1)[0].strip()
    ]

    return len(lines)


def classify_v5c(record):

    try:
        conversation = record["input"]["conversation"]
        summary = record["output"]["summary"]

        if not isinstance(conversation, str):
            return "invalid"

        if not isinstance(summary, str) or not summary.strip():
            return "invalid"

        turns = count_turns(conversation)
        words = len(conversation.split())

        # Little / no useful information
        if turns <= 2 or words <= 10:
            return "little_none"

        # Partial information
        if turns <= 5 or words <= 35:
            return "partial"

        # Complete / rich information
        return "complete"

    except Exception:
        return "invalid"


with open(FILE, "r", encoding="utf-8") as f:

    for line_no, line in enumerate(f, 1):

        if not line.strip():
            continue

        total += 1

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            counts["invalid"] += 1
            continue

        category = classify_v5c(record)
        counts[category] += 1


print("=" * 85)
print("V5C INFORMATION DISTRIBUTION CHECK")
print("=" * 85)

print(f"File          : {FILE}")
print(f"Total records : {total:,}")
print()

for category in ["complete", "partial", "little_none", "invalid"]:

    count = counts[category]
    percentage = (count / total * 100) if total else 0

    print(
        f"{category.replace('_', ' ').title():20}"
        f": {count:8,}  ({percentage:6.2f}%)"
    )


print()
print("=" * 85)
print("EXPECTED V5C DISTRIBUTION")
print("=" * 85)

for category, percentage in EXPECTED.items():

    expected_count = round(total * percentage)

    print(
        f"{category.replace('_', ' ').title():20}"
        f": {expected_count:8,}  ({percentage * 100:6.2f}%)"
    )


print()
print("=" * 85)
print("DIFFERENCE FROM TARGET")
print("=" * 85)

for category in EXPECTED:

    actual = counts[category]
    expected = round(total * EXPECTED[category])
    difference = actual - expected

    print(
        f"{category.replace('_', ' ').title():20}"
        f": {difference:+,}"
    )

print("=" * 85)

V5C INFORMATION DISTRIBUTION CHECK
File          : G:\V5-dataset\V5C\V5C.jsonl
Total records : 64,374

Complete            :   33,342  ( 51.79%)
Partial             :    1,187  (  1.84%)
Little None         :   29,687  ( 46.12%)
Invalid             :      158  (  0.25%)

EXPECTED V5C DISTRIBUTION
Complete            :   51,499  ( 80.00%)
Partial             :    9,656  ( 15.00%)
Little None         :    3,219  (  5.00%)

DIFFERENCE FROM TARGET
Complete            : -18,157
Partial             : -8,469
Little None         : +26,468


In [2]:
import json
import re
from collections import Counter

FILE = r"G:\V5-dataset\V5C\V5C.jsonl"


# ============================================================
# INFORMATION SIGNALS
# ============================================================

# Signals that indicate meaningful factual/task information
INFORMATION_PATTERNS = [

    # Goals / plans / intentions
    r"\b(i|we|he|she|they)\s+(want|need|plan|planned|hope|intend|trying|try|decided|decide)\b",
    r"\b(going to|planning to|would like to|aim to)\b",

    # Problems / issues
    r"\b(problem|issue|trouble|difficulty|difficult|struggling|stuck|concern|worried|stress|stressed)\b",
    r"\b(can't|cannot|unable|failed|failure|broken|not working)\b",

    # Events / actions
    r"\b(started|finished|completed|bought|sold|joined|left|moved|visited|met|called|sent|received|applied|accepted|rejected)\b",

    # Work / study / career
    r"\b(work|job|career|office|project|meeting|interview|report|developer|programming|study|studying|exam|college|school)\b",

    # Relationships / social
    r"\b(friend|partner|family|relationship|wife|husband|girlfriend|boyfriend|brother|sister|parent)\b",

    # Preferences
    r"\b(like|love|prefer|favorite|hate|dislike|enjoy)\b",

    # Time / frequency
    r"\b(today|tomorrow|yesterday|tonight|week|month|year|daily|weekly|monthly|every|often|usually|recently|lately)\b",

    # Quantitative information
    r"\b\d+(?:\.\d+)?\s*(%|percent|kg|km|hours?|days?|weeks?|months?|years?|₹|\$|dollars?)\b",

    # Requests / questions that contain task information
    r"\b(why|how|what|when|where|which|should|could|can you|help me)\b",

    # Emotions / situations
    r"\b(happy|sad|angry|frustrated|excited|confused|afraid|anxious|calm|upset|tired|overwhelmed)\b",

    # Decisions / outcomes
    r"\b(decided|decision|agreed|disagreed|resolved|result|outcome|conclusion)\b",
]


COMPILED_PATTERNS = [
    re.compile(pattern, re.IGNORECASE)
    for pattern in INFORMATION_PATTERNS
]


# ============================================================
# BASIC TEXT PROCESSING
# ============================================================

def clean_text(text):
    if not isinstance(text, str):
        return ""

    return re.sub(r"\s+", " ", text).strip()


def get_turns(conversation):
    if not isinstance(conversation, str):
        return []

    return [
        line.strip()
        for line in conversation.splitlines()
        if line.strip()
    ]


def get_words(text):
    return re.findall(r"\b[\w'-]+\b", text)


# ============================================================
# INFORMATION SCORE
# ============================================================

def information_score(conversation):

    text = clean_text(conversation)

    if not text:
        return 0

    words = get_words(text)
    word_count = len(words)

    turns = get_turns(conversation)
    turn_count = len(turns)

    score = 0

    # --------------------------------------------------------
    # 1. Meaningful length
    # --------------------------------------------------------

    if word_count >= 15:
        score += 1

    if word_count >= 30:
        score += 1

    if word_count >= 60:
        score += 1

    if word_count >= 100:
        score += 1

    # --------------------------------------------------------
    # 2. Number of conversation turns
    # --------------------------------------------------------

    if turn_count >= 4:
        score += 1

    if turn_count >= 8:
        score += 1

    if turn_count >= 15:
        score += 1

    # --------------------------------------------------------
    # 3. Task-relevant information signals
    # --------------------------------------------------------

    matched_signals = 0

    for pattern in COMPILED_PATTERNS:

        if pattern.search(text):
            matched_signals += 1

    # Cap the contribution so repeated words don't dominate
    score += min(matched_signals, 5)

    # --------------------------------------------------------
    # 4. Information diversity
    # --------------------------------------------------------

    unique_words = set(word.lower() for word in words)

    if word_count > 0:

        lexical_diversity = len(unique_words) / word_count

        if lexical_diversity >= 0.45:
            score += 1

        if lexical_diversity >= 0.60:
            score += 1

    # --------------------------------------------------------
    # 5. Multiple substantive user statements
    # --------------------------------------------------------

    user_lines = [
        line for line in turns
        if re.match(
            r"^(user|me|i)\s*:",
            line,
            re.IGNORECASE
        )
    ]

    if len(user_lines) >= 2:
        score += 1

    if len(user_lines) >= 4:
        score += 1

    return score


# ============================================================
# V5C CLASSIFICATION
# ============================================================

def classify_v5c(conversation):

    text = clean_text(conversation)

    if not text:
        return "little_none", 0

    score = information_score(conversation)

    # --------------------------------------------------------
    # Little / None
    # --------------------------------------------------------
    #
    # Almost no task-relevant information.
    #
    if score <= 3:
        return "little_none", score

    # --------------------------------------------------------
    # Partial
    # --------------------------------------------------------
    #
    # Some meaningful information exists, but the
    # conversation is relatively limited.
    #
    if score <= 7:
        return "partial", score

    # --------------------------------------------------------
    # Complete / Rich
    # --------------------------------------------------------

    return "complete", score


# ============================================================
# DATASET SCAN
# ============================================================

counts = Counter()

total = 0

invalid_json = 0
invalid_structure = 0
invalid_output = 0

examples = {
    "complete": [],
    "partial": [],
    "little_none": []
}


with open(FILE, "r", encoding="utf-8") as f:

    for line_no, line in enumerate(f, 1):

        if not line.strip():
            continue

        total += 1

        # ----------------------------------------------------
        # JSON validation
        # ----------------------------------------------------

        try:
            record = json.loads(line)

        except json.JSONDecodeError:
            invalid_json += 1
            continue

        # ----------------------------------------------------
        # Structure validation
        # ----------------------------------------------------

        try:
            conversation = record["input"]["conversation"]
            output = record["output"]

        except (KeyError, TypeError):
            invalid_structure += 1
            continue

        # ----------------------------------------------------
        # V5C output validation
        # ----------------------------------------------------

        if (
            not isinstance(output, dict)
            or not isinstance(output.get("summary"), str)
            or not output.get("summary").strip()
        ):
            invalid_output += 1
            continue

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        category, score = classify_v5c(conversation)

        counts[category] += 1

        # ----------------------------------------------------
        # Save examples
        # ----------------------------------------------------

        if len(examples[category]) < 3:

            examples[category].append({
                "line": line_no,
                "score": score,
                "conversation": conversation,
                "summary": output["summary"]
            })


# ============================================================
# RESULTS
# ============================================================

valid = (
    counts["complete"]
    + counts["partial"]
    + counts["little_none"]
)

print()
print("=" * 90)
print("V5C CONTENT-BASED INFORMATION DISTRIBUTION")
print("=" * 90)

print(f"File                 : {FILE}")
print(f"Total records        : {total:,}")
print(f"Valid V5C records    : {valid:,}")
print()

print(
    f"Complete / Rich      : "
    f"{counts['complete']:8,} "
    f"({counts['complete'] / valid * 100:6.2f}%)"
    if valid else
    "Complete / Rich      : 0"
)

print(
    f"Partial              : "
    f"{counts['partial']:8,} "
    f"({counts['partial'] / valid * 100:6.2f}%)"
    if valid else
    "Partial              : 0"
)

print(
    f"Little / None        : "
    f"{counts['little_none']:8,} "
    f"({counts['little_none'] / valid * 100:6.2f}%)"
    if valid else
    "Little / None        : 0"
)

print()

print(f"Invalid JSON         : {invalid_json:,}")
print(f"Invalid structure    : {invalid_structure:,}")
print(f"Invalid output       : {invalid_output:,}")


# ============================================================
# TARGET COMPARISON
# ============================================================

print()
print("=" * 90)
print("TARGET DISTRIBUTION — V5C")
print("=" * 90)

targets = {
    "complete": 0.80,
    "partial": 0.15,
    "little_none": 0.05
}

for category, percentage in targets.items():

    target_count = round(valid * percentage)
    actual_count = counts[category]
    difference = actual_count - target_count

    name = category.replace("_", " ").title()

    print(
        f"{name:20}"
        f" Target: {target_count:8,}"
        f" | Actual: {actual_count:8,}"
        f" | Difference: {difference:+8,}"
    )


# ============================================================
# EXAMPLES
# ============================================================

print()
print("=" * 90)
print("SAMPLE CLASSIFICATIONS")
print("=" * 90)

for category in ["complete", "partial", "little_none"]:

    print()
    print("-" * 90)
    print(category.upper())
    print("-" * 90)

    for example in examples[category]:

        print(f"\nLine  : {example['line']}")
        print(f"Score : {example['score']}")
        print(f"Conversation:")
        print(example["conversation"][:700])

        print(f"\nSummary:")
        print(example["summary"][:400])


print()
print("=" * 90)
print("CHECK COMPLETE")
print("=" * 90)


V5C CONTENT-BASED INFORMATION DISTRIBUTION
File                 : G:\V5-dataset\V5C\V5C.jsonl
Total records        : 64,374
Valid V5C records    : 64,216

Complete / Rich      :   56,687 ( 88.28%)
Partial              :    7,529 ( 11.72%)
Little / None        :        0 (  0.00%)

Invalid JSON         : 0
Invalid structure    : 156
Invalid output       : 2

TARGET DISTRIBUTION — V5C
Complete             Target:   51,373 | Actual:   56,687 | Difference:   +5,314
Partial              Target:    9,632 | Actual:    7,529 | Difference:   -2,103
Little None          Target:    3,211 | Actual:        0 | Difference:   -3,211

SAMPLE CLASSIFICATIONS

------------------------------------------------------------------------------------------
COMPLETE
------------------------------------------------------------------------------------------

Line  : 1
Score : 9
Conversation:
Colleague: Hey, are you ready for the quarterly budget review meeting at 2 PM?
User: Almost, I'm just finalizing the slide